In [ ]:
!pip install requests beautifulsoup4

In [ ]:
import pandas as pd 
import spacy 
import requests
import json
import re
from bs4 import BeautifulSoup
nlp = spacy.load("en_core_web_sm")
pd.set_option("display.max_rows", 200)
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
#model path
mpath = "/caragiale/Llama-2-7b-chat-hf"
#gpu
# device_map = {'':1}
device_map={"": 2}
#new model path
fn_model = "Llama-2-7b-ovi-fn"

In [ ]:
def inches_to_mm(inch):
    if inch == '-':
        return inch
    if '<' in inch:
        condition, value = inch.split()
        return f"{condition} {float(value) * 25.4}"
    return str(float(inch) * 25.4)

def generate_meteoblue_url(lat, lon):
    # Create the URL
    url = f"https://www.meteoblue.com/en/weather/week/{abs(lat):.3f}N{abs(lon):.3f}E"
    return url

def parse_respone(html_content):
  soup = BeautifulSoup(html_content, 'html.parser')
  weather_data = soup.find('div', class_='col-12 report no-top-padding').find('p').text

  table = soup.find('table', class_='picto three-hourly-view')
  times = table.find('tr', class_='times').text
  times = [item for item in times.split('\n') if item != '']
  day = times[0]
  times = times[1:]

  clouds = [cell.find('div', class_='pictoicon') for cell in table.find('tr', class_='icons').find_all('div', class_='cell')]
  clouds = [cell for cell in clouds if cell is not None]
  clouds = [cell.find('img')['title'] for cell in clouds]

  temperatures = table.find('tr', class_='temperatures').text
  temperatures = [int(item.strip()[:-1]) for item in temperatures.split('\n') if any(char.isdigit() for char in item)][::2]

  temperatures_felt = table.find('tr', class_='windchills').text
  temperatures_felt = [int(item.strip()[:-1]) for item in temperatures_felt.split('\n') if any(char.isdigit() for char in item)]

  wind = table.find('tr', class_='windspeeds').text.split('\n')
  wind = [item.strip() for item in wind if item != ''][1:]
  wind = [item for index, item in enumerate(wind) if (index + 1) % 3 != 0]
  wind_speed = wind[1::2]
  wind_direction = wind[0::2]

  precipitations = table.find('tr', class_='precips').text.split('\n')
  precipitations = [item.strip() for item in precipitations if item != ''][2::]

  precipitations_probability = precipitations[1::2]
  precipitations_size = [inches_to_mm(item) for item in precipitations[2::3]]

  weather_json = {
    "time": times,
    "temperature_real (°F)": temperatures,
    "temperature_felt (°F)": temperatures_felt,
    "wind_speed (km/h)": wind_speed,
    "wind_direction": wind_direction,
    "precipitation_size (mm/3h)": precipitations_size,
    "precipitation_probability": precipitations_probability,
    "cloud_cover": clouds
  }

  return weather_json, weather_data

def clean_text(text):
    # Remove non-ASCII characters
    cleaned_text = re.sub(r'[^\x00-\x7FЋ]+', '', text)
    cleaned_text = re.sub(r'[\[\]]', '', cleaned_text)
    cleaned_text = RE_EMOJI.sub(r'', cleaned_text)

    paragraphs = cleaned_text.split('\n')

    longest_paragraph = ''
    max_length = 0
    
    for paragraph in paragraphs:
        sub_paragraphs = paragraph.split('   ')
        
        for sub_paragraph in sub_paragraphs:
            length = len(sub_paragraph)
            
            if length > max_length:
                max_length = length
                longest_paragraph = sub_paragraph
    
    return longest_paragraph

base_model = AutoModelForCausalLM.from_pretrained(
    mpath,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map=device_map,
)
model = PeftModel.from_pretrained(base_model, fn_model)
model = model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(mpath, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

df = pd.read_csv('worldcities.csv')

RE_EMOJI = re.compile('[\U00010000-\U0010ffff]', flags=re.UNICODE)

In [ ]:
content = ''
while(True):
    content = input("The forecast of which city would you like to know?")

    doc = nlp(content)
    # find city name in input
    city = ''
    for ent in doc.ents:
        if ent.label_ == "GPE":
            city = ent.text
            break
    if content == '':
        print(f'Closing... {city} {content}')
        break
    if city == '':
        print('City not in db')
    # iterate over the rows and check if the city name matches
    for index, row in df.iterrows():
        if row['city'] == city:
            coords = row['lat'],row['lng']

    response = requests.get(generate_meteoblue_url(coords[0], coords[1]))
    print(coords)
    if response.status_code == 200:
        input_json, output = parse_respone(response.text)

    pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=600, return_full_text=False)
    result = pipe(f"<s>[INST] {input_json}[/INST]")[0]['generated_text']
    print(clean_text(result))